# US Revenue Forecast v2 — 단계별 테스트 노트북

**소스 테이블** : `US_IS_from_FMP`  
**저장 테이블** : `us_revenue_forecast_data`  
**예측 모델**  : SARIMA · ETS · Prophet · LSTM · Theta + Ensemble (SARIMA+ETS+Theta 평균)

---

| 셀 번호 | 단계 |
|---------|------|
| Cell 1  | 환경 설정 & 경로 자동 감지 |
| Cell 2  | 모듈 Import |
| Cell 3  | 파라미터 설정 |
| Cell 4  | DB 연결 테스트 |
| Cell 5  | 재무 데이터 추출 함수 정의 |
| Cell 6  | 단일 티커 데이터 추출 테스트 |
| Cell 7  | 예측 함수 정의 |
| Cell 8  | 단일 티커 예측 테스트 |
| Cell 9  | Long-format 변환 함수 정의 |
| Cell 10 | Long-format 변환 테스트 |
| Cell 11 | DB 테이블 생성 & 저장 함수 정의 |
| Cell 12 | 단일 티커 저장 테스트 |
| Cell 13 | 배치 실행 (전체 / 특정 티커 / 구간 지정) |
| Cell 14 | 저장 결과 조회 |

---
### ⚡ 메모리 전략
> **티커 1개씩 즉시 저장** 방식을 채택합니다.  
> 배치(20개 누적 후 저장)는 리스트가 메모리에 쌓여 오히려 OOM 위험이 높습니다.  
> 1개 예측 → 즉시 저장 → `clear_memory()` 호출 순서로 메모리를 최소 상태로 유지합니다.

### 🔢 구간 예측 (2000개 티커 단계적 처리)
> Cell 13 의 `TICKER_START` / `TICKER_END` 변수로 처리 구간을 지정하세요.  
> 예: 0~499 → 500~999 → ... 순서로 끊어서 실행하면 메모리 부담 없이 전체 예측 가능합니다.

## Cell 1 · 환경 설정 & 경로 자동 감지

노트북(Hoyoung_Park) / 데스크탑(82108) 어느 환경에서 실행해도  
`DATA` 폴더를 자동으로 찾아 `sys.path`에 추가합니다.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 후보 프로젝트 루트 (노트북 / 데스크탑) ───────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",          # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",   # 데스크탑
]

def _setup_path() -> str:
    """
    프로젝트 루트(DATA/ 폴더의 부모)를 탐색해 sys.path 에 추가합니다.
    탐색 순서:
      1) __file__ 또는 cwd 기준 상위 경로 중 DATA/ 를 포함하는 첫 번째 경로
      2) _CANDIDATE_ROOTS 에서 실존하는 첫 번째 경로
    """
    try:
        start = Path(__file__).resolve().parent
    except NameError:          # 노트북 환경 — __file__ 없음
        start = Path.cwd()

    # 현재 경로부터 상위로 올라가며 DATA/ 탐색
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root

    # cwd 탐색에서 못 찾으면 후보 경로 시도
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate

    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다.\n"
        "_CANDIDATE_ROOTS 목록을 현재 환경에 맞게 수정하거나 "
        "노트북을 프로젝트 루트 아래에서 실행하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트  : {_ROOT}")
print(f"[확인] DATA 경로     : {os.path.join(_ROOT, 'DATA')}")
print(f"[확인] sys.path[0]   : {sys.path[0]}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로     : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
[확인] sys.path[0]   : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\미국전기업_매출_예측


## Cell 2 · 모듈 Import

`DATA` 폴더 내 세 모듈과 외부 라이브러리를 불러옵니다.

In [2]:
# ── 내부 모듈 (DATA 폴더) ─────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as DEFAULT_TICKER_LIST
from DATA.universal_ts_forecast_function_v2 import (
    forecast_sarima,
    forecast_ets,
    forecast_prophet,
    forecast_lstm,
    forecast_theta,
    infer_freq_alias,
    seasonal_periods_from_freq,
    clear_memory,
)

# ── 외부 라이브러리 ───────────────────────────────────────────
import gc
import traceback
from typing import Optional, List      # Python 3.9 호환 타입 힌트
import numpy as np
import pandas as pd
from datetime import datetime
from sqlalchemy import text
from IPython.display import display

# ── 로그 유틸 (config 에 log 가 없는 경우 자체 정의) ──────────
try:
    from DATA.config import log
except ImportError:
    def log(tag: str, msg: str):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}][{tag}] {msg}")

print(f"[OK] 모든 모듈 Import 완료")
print(f"[OK] DEFAULT_TICKER_LIST 길이: {len(DEFAULT_TICKER_LIST):,}개")


[OK] 모든 모듈 Import 완료
[OK] DEFAULT_TICKER_LIST 길이: 2,000개


## Cell 3 · 파라미터 설정

항목·기간·모델·배치 등 전역 파라미터를 여기서만 수정합니다.

In [3]:
# ════════════════════════════════════════════════════════════
#  ★ 파라미터 — 필요에 따라 이 셀만 수정하세요 ★
# ════════════════════════════════════════════════════════════

# ── 테이블 ────────────────────────────────────────────────
SRC_TABLE  = "US_IS_from_FMP"           # 원본 재무 테이블
DEST_TABLE = "us_revenue_forecast_data" # 예측 결과 저장 테이블

# ── 재무 항목 ─────────────────────────────────────────────
# 예: "sale" (매출) / "opi" (영업이익) / "ni" (순이익) / "ebitda" 등
ITEM       = "sale"

# ── 예측 설정 ─────────────────────────────────────────────
HORIZON    = 8    # 예측 분기 수 (default 8 = 2년)
MIN_OBS    = 28   # 최소 관측 분기 수 (28 = 7년 × 4분기)

# ── 모델 선택 ─────────────────────────────────────────────
ALL_MODELS      = ["SARIMA", "ETS", "Prophet", "LSTM", "Theta"]
ENSEMBLE_MODELS = ["SARIMA", "ETS", "Theta"]   # 앙상블 구성 모델

# ── 예측 실행일 ───────────────────────────────────────────
FORECAST_DATE = datetime.now().strftime("%Y-%m-%d")

# ════════════════════════════════════════════════════════════
print("[파라미터 확인]")
print(f"  SRC_TABLE    = {SRC_TABLE}")
print(f"  DEST_TABLE   = {DEST_TABLE}")
print(f"  ITEM         = {ITEM}")
print(f"  HORIZON      = {HORIZON}분기")
print(f"  MIN_OBS      = {MIN_OBS}개")
print(f"  ALL_MODELS   = {ALL_MODELS}")
print(f"  ENSEMBLE     = {ENSEMBLE_MODELS}")
print(f"  FORECAST_DATE= {FORECAST_DATE}")


[파라미터 확인]
  SRC_TABLE    = US_IS_from_FMP
  DEST_TABLE   = us_revenue_forecast_data
  ITEM         = sale
  HORIZON      = 8분기
  MIN_OBS      = 28개
  ALL_MODELS   = ['SARIMA', 'ETS', 'Prophet', 'LSTM', 'Theta']
  ENSEMBLE     = ['SARIMA', 'ETS', 'Theta']
  FORECAST_DATE= 2026-03-27


## Cell 4 · DB 연결 테스트

In [4]:
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("[OK] DB 연결 성공")
    print(f"     host={db_info.get('host')}  port={db_info.get('port')}  db={db_info.get('database')}")
except Exception as e:
    print(f"[FAIL] DB 연결 실패: {e}")


[OK] DB 연결 성공
     host=192.168.0.230  port=3307  db=investar


## Cell 5 · 재무 데이터 추출 함수 정의

`US_IS_from_FMP` → ticker + item 기준 분기 시계열 추출  

**처리 흐름**
1. ticker / item 기준으로 value, date, period, date_month 추출  
2. 날짜 파싱 및 오름차순 정렬  
3. 월별 중복 제거 (같은 월 → 마지막 행, 단독 행은 보존)  
4. 분기 단위 재집계 (같은 분기 → 마지막 행, 분기말 날짜로 통일)  
5. MIN_OBS 미달 시 ValueError

In [5]:
def fetch_financial_series(
    engine,
    ticker: str,
    item: str   = "sale",
    min_obs: int = 28,
) -> pd.DataFrame:
    """
    US_IS_from_FMP 테이블에서 ticker + item 기준으로 분기 시계열을 추출합니다.

    Parameters
    ----------
    engine  : SQLAlchemy engine
    ticker  : 종목 코드  (예: 'AAPL')
    item    : 재무 항목  (예: 'sale' / 'opi' / 'ni')
    min_obs : 최소 분기 수 — 미달 시 ValueError

    Returns
    -------
    DataFrame  columns: date, report_date, period, date_month, value
    """
    query = text("""
        SELECT
            date,
            report_date,
            period,
            date_month,
            value
        FROM   US_IS_from_FMP
        WHERE  ticker = :ticker
          AND  item   = :item
          AND  value  IS NOT NULL
        ORDER  BY date
    """)

    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"ticker": ticker, "item": item})

    if df.empty:
        raise ValueError(f"[{ticker}] '{item}' 데이터가 DB에 없습니다.")

    # 1. 날짜 변환
    df["date"]        = pd.to_datetime(df["date"],        errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
    df["date_month"]  = pd.to_datetime(df["date_month"],  errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"])

    # 2. 월별 중복 제거
    #    같은 월에 여러 행 → 마지막 행 유지
    #    처음/끝처럼 해당 월에 1건만 있는 행은 그대로 보존
    df["_ym"]  = df["date"].dt.to_period("M")
    dup_mask   = df["_ym"].duplicated(keep=False)

    if dup_mask.any():
        df_single = df[~dup_mask].copy()
        df_dup    = (
            df[dup_mask]
            .groupby("_ym", sort=True)
            .last()
            .reset_index()
        )
        df = (
            pd.concat([df_single, df_dup], ignore_index=True)
            .sort_values("date")
            .reset_index(drop=True)
        )
    df = df.drop(columns=["_ym"])

    # 3. 분기 단위 재집계 → 분기말 날짜로 통일
    df["_q"] = df["date"].dt.to_period("Q")
    df = (
        df.groupby("_q", sort=True)
          .last()
          .reset_index()
    )
    df["date"] = df["_q"].dt.to_timestamp("Q")   # 예: 2023Q4 → 2023-12-31
    df = (
        df.drop(columns=["_q"])
          .sort_values("date")
          .reset_index(drop=True)
    )

    # 4. 최소 관측치 검사
    if len(df) < min_obs:
        raise ValueError(
            f"[{ticker}] '{item}' 관측치 부족: {len(df)}개 < 최소 {min_obs}개"
        )

    return df   # columns: date, report_date, period, date_month, value

print("[OK] fetch_financial_series 함수 정의 완료")


[OK] fetch_financial_series 함수 정의 완료


## Cell 6 · 단일 티커 데이터 추출 테스트

`TEST_TICKER` 를 원하는 티커로 변경해서 테스트하세요.

In [6]:
TEST_TICKER = "AAPL"   # ← 테스트할 티커

try:
    src_df = fetch_financial_series(
        engine, TEST_TICKER, item=ITEM, min_obs=MIN_OBS
    )
    print(f"[OK] {TEST_TICKER} '{ITEM}' 추출 성공: {len(src_df)}분기")
    print(f"     기간: {src_df['date'].iloc[0].date()} ~ {src_df['date'].iloc[-1].date()}")
    display(src_df.tail(8))
except Exception as e:
    print(f"[FAIL] {e}")


[OK] AAPL 'sale' 추출 성공: 45분기
     기간: 2015-03-31 ~ 2026-03-31


,date,report_date,period,date_month,value
37,2024-06-30,2024-06-29,Q3,2024-06-01,8.577700e+10
38,2024-09-30,2024-09-28,Q4,2024-09-01,9.493000e+10
39,2024-12-31,2024-12-28,Q1,2024-12-01,1.243000e+11
40,2025-03-31,2025-03-29,Q2,2025-03-01,9.535900e+10
41,2025-06-30,2025-06-28,Q3,2025-06-01,9.403600e+10
42,2025-09-30,2025-09-27,Q4,2025-09-01,1.024660e+11
43,2025-12-31,2025-12-27,Q1,2025-12-01,1.437560e+11
44,2026-03-31,2025-12-27,Q1,2025-12-01,1.437560e+11


## Cell 7 · 예측 함수 정의

- `make_forecast_index` : 마지막 실제값 다음 분기부터 HORIZON 개 날짜 생성  
- `forecast_one_ticker` : 5개 모델 순차 실행, 메모리 추적 포함

In [7]:
def make_forecast_index(
    last_date: pd.Timestamp,
    horizon: int,
    freq: str = "QE",
) -> pd.DatetimeIndex:
    """
    last_date 다음 분기부터 horizon 개의 날짜 인덱스를 생성합니다.
    freq : infer_freq_alias() 가 반환하는 값 (Q, QE, QS 등)
    """
    _freq = freq if freq else "QE"
    try:
        idx = pd.date_range(
            start   = last_date + pd.tseries.frequencies.to_offset(_freq),
            periods = horizon,
            freq    = _freq,
        )
    except Exception:
        # fallback: 3개월 간격으로 직접 생성
        idx = pd.date_range(
            start   = last_date + pd.DateOffset(months=3),
            periods = horizon,
            freq    = "QE",
        )
    return idx


def forecast_one_ticker(
    y: pd.Series,
    ticker: str,
    horizon: int,
    models: list,
) -> dict:
    """
    단일 티커의 시계열 y 에 대해 지정 모델들로 예측을 수행합니다.

    실제 함수 시그니처 (universal_ts_forecast_function_v2.py 기준):
      forecast_sarima  : (y, forecast_horizon, seasonal_period=int)
      forecast_ets     : (y, forecast_horizon, m=int)
      forecast_prophet : (y, forecast_horizon, m=int)
      forecast_lstm    : (y, forecast_horizon)           ← m 파라미터 없음
      forecast_theta   : (y, forecast_horizon, m=int)

    Parameters
    ----------
    y       : DatetimeIndex 를 가진 분기 시계열 (Series)
    ticker  : 종목 코드 (로그 출력용)
    horizon : 예측 분기 수
    models  : 사용할 모델 목록  ["SARIMA", "ETS", "Prophet", "LSTM", "Theta"]

    Returns
    -------
    dict  {model_name: {"forecast": array, "spec": dict} or {"error": str}}
    """
    import psutil, os
    proc = psutil.Process(os.getpid())

    def _mem_mb():
        return proc.memory_info().rss / 1024 / 1024

    freq    = infer_freq_alias(y.index)
    sp      = seasonal_periods_from_freq(freq)   # 분기=4, 월=12
    results = {}

    # ── 각 함수의 실제 파라미터명에 맞춰 호출 ─────────────────────
    def _call(model_name):
        if model_name == "SARIMA":
            # forecast_sarima(y, forecast_horizon, seasonal_period=)
            return forecast_sarima(y, horizon, seasonal_period=sp)
        elif model_name == "ETS":
            # forecast_ets(y, forecast_horizon, m=)
            return forecast_ets(y, horizon, m=sp)
        elif model_name == "Prophet":
            # forecast_prophet(y, forecast_horizon, m=)
            return forecast_prophet(y, horizon, m=sp)
        elif model_name == "LSTM":
            # forecast_lstm(y, forecast_horizon)  ← m 파라미터 없음
            return forecast_lstm(y, horizon)
        elif model_name == "Theta":
            # forecast_theta(y, forecast_horizon, m=)
            return forecast_theta(y, horizon, m=sp)
        else:
            raise ValueError(f"알 수 없는 모델: {model_name}")

    for model_name in models:
        log(ticker, f"  [{model_name}] 시작  (메모리: {_mem_mb():.1f} MB)")
        try:
            res = _call(model_name)
            results[model_name] = res
            fc_arr = np.asarray(res.get("forecast", []))
            if len(fc_arr) > 0:
                log(ticker, f"  [{model_name}] 완료  첫값={fc_arr[0]:.2e} (메모리: {_mem_mb():.1f} MB)")
            else:
                log(ticker, f"  [{model_name}] 오류응답: {res}")
        except Exception as e:
            log(ticker, f"  [{model_name}] 오류: {e}")
            results[model_name] = {"error": str(e)}
        finally:
            gc.collect()

    return results

print("[OK] make_forecast_index / forecast_one_ticker 함수 정의 완료")
print("  SARIMA  : forecast_sarima(y, forecast_horizon, seasonal_period=sp)")
print("  ETS     : forecast_ets(y, forecast_horizon, m=sp)")
print("  Prophet : forecast_prophet(y, forecast_horizon, m=sp)")
print("  LSTM    : forecast_lstm(y, forecast_horizon)")
print("  Theta   : forecast_theta(y, forecast_horizon, m=sp)")


[OK] make_forecast_index / forecast_one_ticker 함수 정의 완료
  SARIMA  : forecast_sarima(y, forecast_horizon, seasonal_period=sp)
  ETS     : forecast_ets(y, forecast_horizon, m=sp)
  Prophet : forecast_prophet(y, forecast_horizon, m=sp)
  LSTM    : forecast_lstm(y, forecast_horizon)
  Theta   : forecast_theta(y, forecast_horizon, m=sp)


## Cell 8 · 단일 티커 예측 테스트

Cell 6 에서 추출한 `src_df` 를 사용합니다.  
모델별 예측값과 SARIMA 파라미터를 확인하세요.

In [8]:
# Cell 6 에서 src_df 가 정상 추출된 경우에만 실행
y = src_df.set_index("date")["value"].copy()
y.index = pd.DatetimeIndex(y.index)
y.name  = ITEM

print(f"예측 입력 시계열: {len(y)}분기  ({y.index[0].date()} ~ {y.index[-1].date()})")

# 테스트용 모델 — 빠른 확인이 필요하면 ['SARIMA', 'ETS', 'Theta'] 로 축소 가능
TEST_MODELS = ALL_MODELS

forecast_results = forecast_one_ticker(y, TEST_TICKER, HORIZON, TEST_MODELS)
freq             = infer_freq_alias(y.index)
forecast_index   = make_forecast_index(y.index[-1], HORIZON, freq)

print("\n[예측 결과 요약]")
for model_name, res in forecast_results.items():
    if "error" in res:
        print(f"  {model_name:<10}: 오류 → {res['error']}")
    else:
        fc  = np.asarray(res["forecast"])
        msg = f"  {model_name:<10}: {fc.round(0).tolist()}"
        if model_name == "SARIMA" and "spec" in res:
            spec = res["spec"]
            aic  = spec.get("ic_value", "")
            aic_str = f"  AIC={aic:.2f}" if isinstance(aic, float) else ""
            msg += f"  | order={spec.get('order')} seasonal={spec.get('seasonal_order')}{aic_str}"
        print(msg)


예측 입력 시계열: 45분기  (2015-03-31 ~ 2026-03-31)
[AAPL]   [SARIMA] 시작  (메모리: 431.6 MB)
[메모리] forecast_sarima 실행 전: 431.62 MB
[메모리] find_best_sarima_params 실행 전: 431.65 MB
[메모리] find_best_sarima_params 실행 후: 436.88 MB (변화: +5.23 MB)
[메모리] forecast_sarima 실행 후: 436.99 MB (변화: +5.37 MB)
[AAPL]   [SARIMA] 완료  첫값=1.35e+11 (메모리: 437.0 MB)
[AAPL]   [ETS] 시작  (메모리: 437.1 MB)
[메모리] forecast_ets 실행 전: 437.06 MB
[메모리] forecast_ets 실행 후: 437.26 MB (변화: +0.20 MB)
[AAPL]   [ETS] 완료  첫값=1.12e+11 (메모리: 437.3 MB)
[AAPL]   [Prophet] 시작  (메모리: 437.3 MB)


00:24:24 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_prophet 실행 전: 437.26 MB


00:24:24 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 440.32 MB (변화: +3.06 MB)
[AAPL]   [Prophet] 완료  첫값=1.19e+11 (메모리: 440.3 MB)
[AAPL]   [LSTM] 시작  (메모리: 440.3 MB)
[메모리] forecast_lstm 실행 전: 440.32 MB
[메모리] forecast_lstm 실행 후: 1448.62 MB (변화: +1008.30 MB)
[경고] 메모리 사용량이 크게 증가했습니다. 메모리 정리를 권장합니다.
[AAPL]   [LSTM] 완료  첫값=1.03e+11 (메모리: 1448.6 MB)
[AAPL]   [Theta] 시작  (메모리: 1448.6 MB)
[메모리] forecast_theta 실행 전: 1448.62 MB
[메모리] forecast_theta 실행 후: 1448.78 MB (변화: +0.16 MB)
[AAPL]   [Theta] 완료  첫값=1.15e+11 (메모리: 1448.8 MB)

[예측 결과 요약]
  SARIMA    : [135238172626.0, 145630469496.0, 190965969156.0, 168115103284.0, 159595361653.0, 171899902806.0, 222578777791.0, 198600825393.0]  | order=(1, 0, 0) seasonal=(1, 0, 1, 4)  AIC=-49.16
  ETS       : [111600659045.0, 119882190949.0, 175206371709.0, 138591237537.0, 120227487135.0, 129149188664.0, 188749976755.0, 149304460839.0]
  Prophet   : [118785779479.0, 120376186029.0, 121966592578.0, 123522425072.0, 125095544593.0, 126685951143.0, 128276357692.0, 129849477213.0]
  LSTM

## Cell 9 · Long-format 변환 함수 정의

actual + forecast(각 모델) + Ensemble → 하나의 long-format DataFrame

**저장 컬럼**

| 컬럼 | 설명 |
|------|------|
| ticker | 종목 코드 |
| item | 재무 항목 |
| date | 기준일(분기말) |
| period | 회계 분기(Q1~Q4/FY) — actual 행만 |
| date_month | 해당 분기 기간 — actual 행만 |
| data_type | `actual` / `forecast` |
| model | actual / SARIMA / ETS / Prophet / LSTM / Theta / Ensemble |
| value | 수치값 |
| forecast_date | 예측 실행일 |
| sarima_order | SARIMA (p,d,q) — SARIMA 행만 |
| sarima_seasonal_order | SARIMA (P,D,Q,m) — SARIMA 행만 |
| sarima_ic_value | SARIMA 최적 AIC — SARIMA 행만 |
| created_at | 레코드 생성 시각 |

In [9]:
def build_long_df(
    ticker: str,
    item: str,
    src_df: pd.DataFrame,
    forecast_results: dict,
    forecast_index: pd.DatetimeIndex,
    forecast_date: str,
    ensemble_models: list = None,
) -> pd.DataFrame:
    """
    실제값(actual) + 예측값(각 모델) + 앙상블 → long-format DataFrame.

    Parameters
    ----------
    ticker           : 종목 코드
    item             : 재무 항목
    src_df           : fetch_financial_series() 반환 DataFrame
    forecast_results : forecast_one_ticker() 반환 dict
    forecast_index   : 예측 날짜 DatetimeIndex
    forecast_date    : 예측 실행일 (str)
    ensemble_models  : 앙상블 구성 모델 목록 (None → ENSEMBLE_MODELS 전역 변수 사용)
    """
    if ensemble_models is None:
        ensemble_models = ENSEMBLE_MODELS

    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows    = []

    # ── actual 행 ────────────────────────────────────────────
    for _, row in src_df.iterrows():
        rows.append({
            "ticker"                : ticker,
            "item"                  : item,
            "date"                  : row["date"].strftime("%Y-%m-%d"),
            "period"                : row.get("period"),
            "date_month"            : (
                row["date_month"].strftime("%Y-%m-%d")
                if pd.notna(row.get("date_month")) else None
            ),
            "data_type"             : "actual",
            "model"                 : "actual",
            "value"                 : round(float(row["value"]), 6),
            "forecast_date"         : forecast_date,
            "sarima_order"          : None,
            "sarima_seasonal_order" : None,
            "sarima_ic_value"       : None,
            "created_at"            : now_str,
        })

    # ── forecast 행 ──────────────────────────────────────────
    ensemble_bucket = {}   # { date_str : [val, ...] }

    for model_name, res in forecast_results.items():
        if "error" in res or "forecast" not in res:
            continue

        fc_arr = np.asarray(res["forecast"])
        spec   = res.get("spec", {})

        sarima_order    = None
        sarima_seasonal = None
        sarima_ic       = None
        if model_name == "SARIMA":
            sarima_order    = str(spec.get("order",          ""))
            sarima_seasonal = str(spec.get("seasonal_order", ""))
            raw_ic = spec.get("ic_value")
            if raw_ic is not None:
                try:
                    v = float(raw_ic)
                    sarima_ic = round(v, 4) if np.isfinite(v) else None
                except (TypeError, ValueError):
                    pass

        for i, dt in enumerate(forecast_index):
            if i >= len(fc_arr):
                break
            val    = float(fc_arr[i])
            dt_str = dt.strftime("%Y-%m-%d")

            rows.append({
                "ticker"                : ticker,
                "item"                  : item,
                "date"                  : dt_str,
                "period"                : None,
                "date_month"            : None,
                "data_type"             : "forecast",
                "model"                 : model_name,
                "value"                 : round(val, 6),
                "forecast_date"         : forecast_date,
                "sarima_order"          : sarima_order,
                "sarima_seasonal_order" : sarima_seasonal,
                "sarima_ic_value"       : sarima_ic,
                "created_at"            : now_str,
            })

            if model_name in ensemble_models:
                ensemble_bucket.setdefault(dt_str, []).append(val)

    # ── Ensemble 행 (SARIMA + ETS + Theta 평균) ─────────────
    for dt_str, vals in ensemble_bucket.items():
        rows.append({
            "ticker"                : ticker,
            "item"                  : item,
            "date"                  : dt_str,
            "period"                : None,
            "date_month"            : None,
            "data_type"             : "forecast",
            "model"                 : "Ensemble",
            "value"                 : round(float(np.mean(vals)), 6),
            "forecast_date"         : forecast_date,
            "sarima_order"          : None,
            "sarima_seasonal_order" : None,
            "sarima_ic_value"       : None,
            "created_at"            : now_str,
        })

    return pd.DataFrame(rows)

print("[OK] build_long_df 함수 정의 완료")


[OK] build_long_df 함수 정의 완료


## Cell 10 · Long-format 변환 테스트

In [10]:
long_df = build_long_df(
    ticker           = TEST_TICKER,
    item             = ITEM,
    src_df           = src_df,
    forecast_results = forecast_results,
    forecast_index   = forecast_index,
    forecast_date    = FORECAST_DATE,
)

print(f"[OK] long_df 생성: {len(long_df)}행")
print("\n모델별 행 수:")
display(long_df.groupby(["data_type", "model"]).size().reset_index(name="rows"))
print("\n샘플 (forecast 상위 5행):")
display(long_df[long_df["data_type"] == "forecast"].head())


[OK] long_df 생성: 93행

모델별 행 수:


,data_type,model,rows
0,actual,actual,45
1,forecast,ETS,8
2,forecast,Ensemble,8
3,forecast,LSTM,8
4,forecast,Prophet,8
5,forecast,SARIMA,8
6,forecast,Theta,8



샘플 (forecast 상위 5행):


,ticker,item,date,period,date_month,data_type,model,value,forecast_date,sarima_order,sarima_seasonal_order,sarima_ic_value,created_at
45,AAPL,sale,2026-06-30,None,None,forecast,SARIMA,1.352382e+11,2026-03-27,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-27 00:24:33
46,AAPL,sale,2026-09-30,None,None,forecast,SARIMA,1.456305e+11,2026-03-27,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-27 00:24:33
47,AAPL,sale,2026-12-31,None,None,forecast,SARIMA,1.909660e+11,2026-03-27,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-27 00:24:33
48,AAPL,sale,2027-03-31,None,None,forecast,SARIMA,1.681151e+11,2026-03-27,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-27 00:24:33
49,AAPL,sale,2027-06-30,None,None,forecast,SARIMA,1.595954e+11,2026-03-27,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-27 00:24:33


## Cell 11 · DB 테이블 생성 & 저장 함수 정의

**중복 판정 기준** : `(ticker, item, date, model, forecast_date)`  
→ 이미 존재하는 행은 건드리지 않고, 신규 행만 INSERT

In [11]:
# ── 테이블 CREATE (최초 1회) ──────────────────────────────────
CREATE_TABLE_SQL = f"""
CREATE TABLE IF NOT EXISTS `{DEST_TABLE}` (
    id                    BIGINT       NOT NULL AUTO_INCREMENT,
    ticker                VARCHAR(20)  NOT NULL,
    item                  VARCHAR(30)  NOT NULL,
    date                  DATE         NOT NULL,
    period                VARCHAR(10)  DEFAULT NULL,
    date_month            DATE         DEFAULT NULL,
    data_type             VARCHAR(10)  NOT NULL COMMENT 'actual / forecast',
    model                 VARCHAR(20)  NOT NULL,
    value                 DOUBLE       DEFAULT NULL,
    forecast_date         DATE         NOT NULL,
    sarima_order          VARCHAR(30)  DEFAULT NULL,
    sarima_seasonal_order VARCHAR(30)  DEFAULT NULL,
    sarima_ic_value       DOUBLE       DEFAULT NULL,
    created_at            DATETIME     DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id),
    UNIQUE KEY uq_main (ticker, item, date, model, forecast_date),
    INDEX idx_ticker      (ticker),
    INDEX idx_forecast_dt (forecast_date),
    INDEX idx_model       (model)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

def ensure_table(engine):
    """저장 테이블이 없으면 생성합니다."""
    with engine.begin() as conn:
        conn.execute(text(CREATE_TABLE_SQL))
    print(f"[OK] 테이블 '{DEST_TABLE}' 준비 완료")


def save_to_db(engine, long_df: pd.DataFrame, dest_table: str = DEST_TABLE) -> int:
    """
    long_df 를 DB 에 저장합니다.
    - 중복 기준: (ticker, item, date, model, forecast_date)
    - 기존 행 유지 + 신규 행만 INSERT

    Returns
    -------
    int  : 실제 삽입된 신규 행 수
    """
    if long_df is None or long_df.empty:
        return 0

    # ── 1. 기존 키 조회 ────────────────────────────────────
    ticker      = long_df["ticker"].iloc[0]
    item        = long_df["item"].iloc[0]
    fc_date_val = long_df["forecast_date"].iloc[0]

    check_sql = text(f"""
        SELECT CONCAT(ticker,'|',item,'|',date,'|',model,'|',forecast_date) AS uq_key
        FROM   `{dest_table}`
        WHERE  ticker        = :ticker
          AND  item          = :item
          AND  forecast_date = :fc_date
    """)

    with engine.connect() as conn:
        existing = pd.read_sql(
            check_sql, conn,
            params={"ticker": ticker, "item": item, "fc_date": fc_date_val}
        )
    existing_keys = set(existing["uq_key"].tolist()) if not existing.empty else set()

    # ── 2. 신규 행 필터링 ──────────────────────────────────
    new_df = long_df[
        ~long_df.apply(
            lambda r: f"{r['ticker']}|{r['item']}|{r['date']}|{r['model']}|{r['forecast_date']}"
            in existing_keys,
            axis=1,
        )
    ].copy()

    if new_df.empty:
        log(ticker, f"  [DB] 신규 행 없음 — 스킵")
        return 0

    # ── 3. INSERT ─────────────────────────────────────────
    new_df.to_sql(
        name       = dest_table,
        con        = engine,
        if_exists  = "append",
        index      = False,
        chunksize  = 500,
        method     = "multi",
    )
    log(ticker, f"  [DB] {len(new_df)}행 저장 완료")
    return len(new_df)

print("[OK] ensure_table / save_to_db 함수 정의 완료")


[OK] ensure_table / save_to_db 함수 정의 완료


## Cell 12 · 단일 티커 저장 테스트

In [12]:
# 테이블 생성 (최초 1회)
ensure_table(engine)

# Cell 10 의 long_df 저장
inserted = save_to_db(engine, long_df)
print(f"[OK] {TEST_TICKER} 저장 완료 — 삽입: {inserted}행")

# 저장 확인
with engine.connect() as conn:
    chk = pd.read_sql(
        text(f"""
            SELECT model, data_type, COUNT(*) AS cnt
            FROM   `{DEST_TABLE}`
            WHERE  ticker = :tk AND forecast_date = :fd
            GROUP  BY model, data_type
            ORDER  BY model
        """),
        conn,
        params={"tk": TEST_TICKER, "fd": FORECAST_DATE},
    )
display(chk)


[OK] 테이블 'us_revenue_forecast_data' 준비 완료
[AAPL]   [DB] 93행 저장 완료
[OK] AAPL 저장 완료 — 삽입: 93행


,model,data_type,cnt
0,actual,actual,45
1,Ensemble,forecast,8
2,ETS,forecast,8
3,LSTM,forecast,8
4,Prophet,forecast,8
5,SARIMA,forecast,8
6,Theta,forecast,8


## Cell 13 · 배치 실행 (전체 / 특정 티커 / 구간 지정)

### 실행 모드 선택

| 변수 | 설명 |
|------|------|
| `RUN_TICKERS` | `None` → DEFAULT_TICKER_LIST 전체 / `["AAPL", ...]` → 특정 티커만 |
| `TICKER_START` | 리스트 슬라이싱 시작 인덱스 (0부터, `None` = 처음) |
| `TICKER_END`   | 리스트 슬라이싱 끝 인덱스 (None = 끝까지) |
| `RUN_MODELS`   | 사용할 모델 목록 |

### 구간 예시
```python
TICKER_START, TICKER_END = 0,   500   # 1~500번째 티커
TICKER_START, TICKER_END = 500, 1000  # 501~1000번째 티커
TICKER_START, TICKER_END = None, None # 전체
```

### 메모리 전략
> 티커 1개 예측 → 즉시 DB 저장 → `clear_memory()` 호출  
> 이 방식이 배치(20개 누적) 방식보다 피크 메모리가 낮고 중단 시 손실도 최소화됩니다.

In [13]:
`1          # ════════════════════════════════════════════════════════════
#  배치 설정 — 여기를 수정하세요
# ════════════════════════════════════════════════════════════

# ── 특정 티커 지정 (None 이면 아래 구간/전체 사용) ────────────
# Optional[list] = Python 3.9 호환 (3.10+ 의 list | None 대신 사용)
RUN_TICKERS = None          # type: Optional[list]
# RUN_TICKERS = ["AAPL", "MSFT", "NVDA"]   # 특정 티커만

# ── 전체 리스트 구간 지정 (RUN_TICKERS=None 일 때 적용) ──────
TICKER_START = 1500            # type: Optional[int]  # 시작 인덱스 (0부터)
TICKER_END   = 2000          # type: Optional[int]  # 끝 인덱스 (exclusive, None=끝까지)
# 예: 0~499   → START=0,   END=500
# 예: 500~999 → START=500, END=1000
# 예: 전체    → START=None, END=None

# ── 모델 / 항목 / 예측 기간 ───────────────────────────────────
RUN_MODELS  = ALL_MODELS   # 또는 ["SARIMA", "ETS", "Theta"]  (빠른 실행)
RUN_ITEM    = ITEM
RUN_HORIZON = HORIZON
RUN_MIN_OBS = MIN_OBS

# ════════════════════════════════════════════════════════════
#  실행 대상 티커 목록 결정
# ════════════════════════════════════════════════════════════
if RUN_TICKERS is not None:
    tickers = RUN_TICKERS
    print(f"[모드] 특정 티커 지정: {tickers}")
else:
    tickers = DEFAULT_TICKER_LIST[TICKER_START:TICKER_END]
    _s = TICKER_START if TICKER_START is not None else 0
    _e = TICKER_END   if TICKER_END   is not None else len(DEFAULT_TICKER_LIST)
    print(f"[모드] 구간 실행: index {_s} ~ {_e-1}  ({len(tickers)}개)")

total = len(tickers)

# ════════════════════════════════════════════════════════════
#  배치 실행
# ════════════════════════════════════════════════════════════
ensure_table(engine)

success, skipped, errored = 0, 0, 0
skip_list, error_list     = [], []

log("BATCH", "=" * 70)
log("BATCH", f"시작  | 티커 {total}개 | 항목: {RUN_ITEM} | 예측기간: {RUN_HORIZON}분기")
log("BATCH", f"모델  : {RUN_MODELS}")
log("BATCH", f"예측일: {FORECAST_DATE} | min_obs: {RUN_MIN_OBS}")
log("BATCH", "=" * 70)

for i, ticker in enumerate(tickers, 1):
    pct = i / total * 100
    log("PROGRESS", f"[{i:>4}/{total}] ({pct:5.1f}%)  >>  {ticker}")

    # ── STEP 1 : 데이터 추출 ──────────────────────────────
    try:
        _src_df = fetch_financial_series(
            engine, ticker, RUN_ITEM, RUN_MIN_OBS
        )
    except Exception as e:
        log(ticker, f"[SKIP] {e}")
        skipped += 1
        skip_list.append(ticker)
        continue

    _y = _src_df.set_index("date")["value"].copy()
    _y.index = pd.DatetimeIndex(_y.index)
    _y.name  = RUN_ITEM
    log(ticker, f"  {len(_y)}분기 | {_y.index[0].date()} ~ {_y.index[-1].date()}")

    # ── STEP 2 : 예측 ─────────────────────────────────────
    try:
        _fc_results = forecast_one_ticker(_y, ticker, RUN_HORIZON, RUN_MODELS)
    except Exception as e:
        log(ticker, f"[ERROR] 예측: {e}")
        traceback.print_exc()
        errored += 1
        error_list.append(ticker)
        del _src_df, _y
        clear_memory()
        continue

    # ── STEP 3 : 예측 인덱스 생성 ────────────────────────
    _freq     = infer_freq_alias(_y.index)
    _fc_index = make_forecast_index(_y.index[-1], RUN_HORIZON, _freq)

    # ── STEP 4 : Long-format 변환 ─────────────────────────
    try:
        _ldf = build_long_df(
            ticker           = ticker,
            item             = RUN_ITEM,
            src_df           = _src_df,
            forecast_results = _fc_results,
            forecast_index   = _fc_index,
            forecast_date    = FORECAST_DATE,
        )
    except Exception as e:
        log(ticker, f"[ERROR] Long-format 변환: {e}")
        errored += 1
        error_list.append(ticker)
        del _src_df, _y, _fc_results
        clear_memory()
        continue

    # ── STEP 5 : DB 저장 (1개씩 즉시 저장 — 메모리 최소화) ─
    try:
        save_to_db(engine, _ldf)
        success += 1
    except Exception as e:
        log(ticker, f"[ERROR] DB 저장: {e}")
        errored += 1
        error_list.append(ticker)

    # ── STEP 6 : 메모리 해제 ─────────────────────────────
    del _src_df, _y, _fc_results, _ldf
    clear_memory()

# ── 요약 ──────────────────────────────────────────────────
log("BATCH", "=" * 70)
log("BATCH", f"완료 | 성공: {success}  스킵: {skipped}  오류: {errored}  합계: {total}")
if skip_list:  log("BATCH", f"스킵 티커  : {skip_list}")
if error_list: log("BATCH", f"오류 티커  : {error_list}")
log("BATCH", "=" * 70)


[메모리] find_best_sarima_params 실행 후: 1541.41 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 후: 1541.41 MB (변화: +0.00 MB)
[AAT]   [SARIMA] 완료  첫값=1.11e+08 (메모리: 1541.4 MB)
[AAT]   [ETS] 시작  (메모리: 1541.4 MB)
[메모리] forecast_ets 실행 전: 1541.41 MB
[메모리] forecast_ets 실행 후: 1541.41 MB (변화: +0.01 MB)
[AAT]   [ETS] 완료  첫값=1.11e+08 (메모리: 1541.4 MB)
[AAT]   [Prophet] 시작  (메모리: 1541.4 MB)
[메모리] forecast_prophet 실행 전: 1541.41 MB


00:43:11 - cmdstanpy - INFO - Chain [1] start processing
00:43:11 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 1541.46 MB (변화: +0.05 MB)
[AAT]   [Prophet] 완료  첫값=1.19e+08 (메모리: 1541.5 MB)
[AAT]   [LSTM] 시작  (메모리: 1541.5 MB)
[메모리] forecast_lstm 실행 전: 1541.46 MB
[메모리] forecast_lstm 실행 후: 1541.68 MB (변화: +0.21 MB)
[AAT]   [LSTM] 완료  첫값=1.19e+08 (메모리: 1541.7 MB)
[AAT]   [Theta] 시작  (메모리: 1541.7 MB)
[메모리] forecast_theta 실행 전: 1541.68 MB
[메모리] forecast_theta 실행 후: 1541.68 MB (변화: +0.00 MB)
[AAT]   [Theta] 완료  첫값=1.10e+08 (메모리: 1541.7 MB)
[AAT]   [DB] 93행 저장 완료
[PROGRESS] [  56/500] ( 11.2%)  >>  CRTO
[CRTO]   45분기 | 2015-03-31 ~ 2026-03-31
[CRTO]   [SARIMA] 시작  (메모리: 1541.7 MB)
[메모리] forecast_sarima 실행 전: 1541.68 MB
[메모리] find_best_sarima_params 실행 전: 1541.68 MB
[메모리] find_best_sarima_params 실행 후: 1541.68 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 후: 1541.68 MB (변화: +0.00 MB)
[CRTO]   [SARIMA] 완료  첫값=5.54e+08 (메모리: 1541.7 MB)
[CRTO]   [ETS] 시작  (메모리: 1541.7 MB)
[메모리] forecast_ets 실행 전: 1541.68 MB
[메모리] forecast_ets 실행 후: 1541.68 MB (변화: +0.00 MB)
[CRTO]   [ETS] 완료  첫값=5.1

00:43:33 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_prophet 실행 전: 1541.68 MB


...

[메모리] forecast_prophet 실행 후: 1599.39 MB (변화: +0.03 MB)
[LOCO]   [Prophet] 완료  첫값=1.24e+08 (메모리: 1599.4 MB)
[LOCO]   [LSTM] 시작  (메모리: 1599.4 MB)
[메모리] forecast_lstm 실행 전: 1599.39 MB
[메모리] forecast_lstm 실행 후: 1600.11 MB (변화: +0.72 MB)
[LOCO]   [LSTM] 완료  첫값=1.20e+08 (메모리: 1600.1 MB)
[LOCO]   [Theta] 시작  (메모리: 1600.1 MB)
[메모리] forecast_theta 실행 전: 1600.11 MB
[메모리] forecast_theta 실행 후: 1600.11 MB (변화: +0.00 MB)
[LOCO]   [Theta] 완료  첫값=1.28e+08 (메모리: 1600.1 MB)
[LOCO]   [DB] 92행 저장 완료
[PROGRESS] [ 500/500] (100.0%)  >>  BNTC
[BNTC]   45분기 | 2015-03-31 ~ 2026-03-31
[BNTC]   [SARIMA] 시작  (메모리: 1600.1 MB)
[메모리] forecast_sarima 실행 전: 1600.11 MB
[메모리] find_best_sarima_params 실행 전: 1600.11 MB
[메모리] find_best_sarima_params 실행 후: 1600.11 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 후: 1600.11 MB (변화: +0.00 MB)
[BNTC]   [SARIMA] 완료  첫값=-6.71e+05 (메모리: 1600.1 MB)
[BNTC]   [ETS] 시작  (메모리: 1600.1 MB)
[메모리] forecast_ets 실행 전: 1600.11 MB
[메모리] forecast_ets 실행 후: 1600.11 MB (변화: +0.00 MB)
[BNTC]   [ETS] 완료 

03:10:41 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_prophet 실행 전: 1600.11 MB


03:10:41 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 1600.11 MB (변화: +0.01 MB)
[BNTC]   [Prophet] 완료  첫값=1.79e+04 (메모리: 1600.1 MB)
[BNTC]   [LSTM] 시작  (메모리: 1600.1 MB)
[메모리] forecast_lstm 실행 전: 1600.11 MB
[메모리] forecast_lstm 실행 후: 1598.36 MB (변화: -1.75 MB)
[BNTC]   [LSTM] 완료  첫값=3.74e+05 (메모리: 1598.4 MB)
[BNTC]   [Theta] 시작  (메모리: 1598.4 MB)
[메모리] forecast_theta 실행 전: 1598.36 MB
[메모리] forecast_theta 실행 후: 1598.36 MB (변화: +0.00 MB)
[BNTC]   [Theta] 완료  첫값=-5.20e+03 (메모리: 1598.4 MB)
[BNTC]   [DB] 93행 저장 완료
[BATCH] ======================================================================
[BATCH] 완료 | 성공: 483  스킵: 17  오류: 0  합계: 500
[BATCH] 스킵 티커  : ['MBX', 'SWIR', 'EZT', 'PVLA', 'WBI', 'LOT', 'HDL', 'TCRZ', 'BTX', 'RZLV', 'BGM', 'AIC', 'TBN', 'ODV', 'KOR', 'FVR', 'SMC']
[BATCH] ======================================================================


## Cell 14 · 저장 결과 조회

배치 완료 후 DB 에 저장된 결과를 확인합니다.

In [14]:
# ── 오늘 예측된 티커 × 모델별 요약 ──────────────────────────
with engine.connect() as conn:
    summary_df = pd.read_sql(
        text(f"""
            SELECT
                ticker,
                item,
                model,
                MIN(date)  AS date_from,
                MAX(date)  AS date_to,
                COUNT(*)   AS row_count,
                forecast_date
            FROM   `{DEST_TABLE}`
            WHERE  forecast_date = :fd
            GROUP  BY ticker, item, model, forecast_date
            ORDER  BY ticker, model
        """),
        conn,
        params={"fd": FORECAST_DATE},
    )

print(f"[오늘({FORECAST_DATE}) 예측 저장 결과: {len(summary_df)}건]")
display(summary_df)


OperationalError: (pymysql.err.OperationalError) (2013, 'Lost connection to MySQL server during query ([WinError 10053] 현재 연결은 사용자의 호스트 시스템의 소프트웨어의 의해 중단되었습니다)')
[SQL: 
            SELECT
                ticker,
                item,
                model,
                MIN(date)  AS date_from,
                MAX(date)  AS date_to,
                COUNT(*)   AS row_count,
                forecast_date
            FROM   `us_revenue_forecast_data`
            WHERE  forecast_date = %(fd)s
            GROUP  BY ticker, item, model, forecast_date
            ORDER  BY ticker, model
        ]
[parameters: {'fd': '2026-03-27'}]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [15]:
# ── 전체 DB 저장 통계 ─────────────────────────────────────
with engine.connect() as conn:
    total_stat = pd.read_sql(
        text(f"""
            SELECT
                forecast_date,
                COUNT(DISTINCT ticker) AS ticker_cnt,
                COUNT(DISTINCT model)  AS model_cnt,
                COUNT(*)               AS total_rows
            FROM   `{DEST_TABLE}`
            GROUP  BY forecast_date
            ORDER  BY forecast_date DESC
            LIMIT  10
        """),
        conn,
    )

print("[전체 DB 저장 현황 (최근 10개 예측일)]")
display(total_stat)


[전체 DB 저장 현황 (최근 10개 예측일)]


,forecast_date,ticker_cnt,model_cnt,total_rows
0,2026-03-27,484,7,44964
1,2026-03-26,688,7,63891
2,2026-03-25,1733,7,160980
3,2026-03-24,1170,7,108691
